# Assessing Overlap in NISAR and ESA BIOMASS Datasets


Date: February 4, 2026  
Updated: September 8, 2026

Authors: Harshini Girish (UAH), Rajat Shinde (UAH), Alex Mandel (Development Seed), Samantha Niemoeller (JPL)

Description: This notebook queries **NISAR Level-2 GCOV PROVISIONAL** granules using `earthaccess` and ESA BIOMASS satellite items using the ESA MAAP STAC API (for example, `BiomassLevel1b`) for a chosen AOI and time range. It converts the returned metadata footprints to GeoDataFrames and plots them on a single interactive Folium map as two toggleable layers. An overlap workflow then identifies where NISAR and BIOMASS footprints intersect to support downstream data-fusion workflows.


## Run This Notebook

To access and run this tutorial within MAAP's Algorithm Development Environment (ADE), please refer to the ["Getting started with the MAAP"](https://docs.maap-project.org/en/latest/getting_started/getting_started.html) section of our documentation.

Disclaimer: it is highly recommended to run a tutorial within MAAP's ADE, which already includes packages specific to MAAP, such as maap-py. Running the tutorial outside of the MAAP ADE may lead to errors. Additionally, it is recommended to use the `Pangeo` workspace within the ADE, since certain packages relevant to this tutorial are already installed.

## Additional Resources
- [NISAR Mission](https://nisar.jpl.nasa.gov/)
- [NISAR Data User Guide — Earthaccess](https://nisar-docs.asf.alaska.edu/earthaccess/)
- [BIOMASS](https://docs.maap-project.org/en/develop/science/ESA_CCI/ESA_CCI_V5_Token_Access.html)


## Import and Install Packages

In [1]:
import earthaccess
import pystac_client
import geopandas as gpd
import pandas as pd
import folium
from folium import GeoJson
from urllib3.util.retry import Retry
from pystac_client.stac_api_io import StacApiIO


## Inputs

This section defines the parameters used to search both catalogs and compute spatial overlaps.

- **BBOX** defines the area of interest as **(min_lon, min_lat, max_lon, max_lat)** and is used to spatially filter both datasets. For routine use, replace the global example with the smallest AOI needed for your analysis.
- **NISAR_DT** defines the NISAR temporal search range as a `(start_date, end_date)` tuple. PROVISIONAL NISAR products are available for acquisitions beginning June 17, 2026.
- **NISAR_SHORT_NAME** is the Earthaccess/CMR short name for the current Level-2 GCOV PROVISIONAL collection: `NISAR_L2_GCOV_PROVISIONAL_V1`.
- **BIOMASS_DT** sets the datetime range for the BIOMASS STAC search.
- **BIOMASS_STAC_URL** is the ESA MAAP STAC endpoint used to query BIOMASS items.
- **BIOMASS_COLLECTION** is the BIOMASS collection name used in the STAC search (`BiomassLevel1b`).
- **BIOMASS_MAX_ITEMS** caps the number of BIOMASS footprints returned so a broad query does not require excessive STAC pagination.

Only metadata footprints are queried here; the NISAR and BIOMASS raster data files are not downloaded.


In [2]:
# Common query parameters (edit to your AOI/time window)
# A smaller AOI is strongly recommended. The global BBOX is retained only as a generic example.
BBOX = [-180, -90, 180, 90]                    # [min_lon, min_lat, max_lon, max_lat]

# NISAR PROVISIONAL products are available for acquisitions beginning 2026-06-17.
NISAR_DT = ("2026-06-17", "2026-09-08")
NISAR_SHORT_NAME = "NISAR_L2_GCOV_PROVISIONAL_V1"

BIOMASS_DT = "2024-01-01/.."                    # adjust if needed
BIOMASS_STAC_URL = "https://catalog.maap.eo.esa.int/catalogue/"
BIOMASS_COLLECTION = "BiomassLevel1b"
BIOMASS_MAX_ITEMS = 200                         # keep broad STAC searches manageable


## Query NISAR with Earthaccess and BIOMASS with STAC


### 1) NISAR GCOV PROVISIONAL data


This section queries the current **NISAR Level-2 GCOV PROVISIONAL** collection through `earthaccess` using the notebook's bounding box and temporal range. `earthaccess.search_data()` returns CMR granule metadata; no HDF5 data files are downloaded.

Each Earthaccess `DataGranule` exposes its CMR footprint through the GeoJSON-compatible `__geo_interface__`. The code converts those footprints into `gdf_nisar`, preserving the NISAR granule identifier for mapping and later spatial joins.


In [3]:
nisar_granules = earthaccess.search_data(
    short_name=NISAR_SHORT_NAME,
    bounding_box=tuple(BBOX),
    temporal=NISAR_DT,
    count=500,
)

print("NISAR granules returned:", len(nisar_granules))

# Convert Earthaccess DataGranules -> GeoDataFrame using their CMR footprints.
nisar_features = []
for granule in nisar_granules:
    umm = granule.get("umm", {})
    granule_id = umm.get("GranuleUR") or granule.get("meta", {}).get("concept-id")

    try:
        geometry = granule.__geo_interface__
    except (KeyError, ValueError):
        # Skip metadata records that do not contain a usable horizontal footprint.
        continue

    nisar_features.append({
        "type": "Feature",
        "geometry": geometry,
        "properties": {
            "nisar_id": granule_id,
        },
    })

if not nisar_features:
    raise ValueError(
        "No NISAR granules with usable footprints were returned. "
        "Check BBOX and NISAR_DT."
    )

gdf_nisar = gpd.GeoDataFrame.from_features(nisar_features, crs="EPSG:4326")
gdf_nisar = gdf_nisar[["nisar_id", "geometry"]]
gdf_nisar.head()


NISAR granules returned: 500


/srv/conda/envs/notebook/lib/python3.13/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


,nisar_id,geometry
0,NISAR_L2_PR_GCOV_023_040_D_124_4005_SHSH_A_202...,"MULTIPOLYGON (((-103.79052 -71.31863, -104.850..."
1,NISAR_L2_PR_GCOV_023_040_D_125_4005_SHSH_A_202...,"MULTIPOLYGON (((-104.73943 -72.34834, -107.090..."
2,NISAR_L2_PR_GCOV_023_040_D_126_4005_SHSH_A_202...,"MULTIPOLYGON (((-106.95305 -74.39485, -109.855..."
3,NISAR_L2_PR_GCOV_023_040_D_127_4005_SHSH_A_202...,"MULTIPOLYGON (((-109.68257 -76.42305, -113.520..."
4,NISAR_L2_PR_GCOV_023_040_D_128_4005_SHSH_A_202...,"MULTIPOLYGON (((-113.28891 -78.4149, -118.3023..."


### 2) ESA BIOMASS

This section connects to the ESA MAAP STAC endpoint (`https://catalog.maap.eo.esa.int/catalogue/`) and searches the `BiomassLevel1b` collection using the notebook's temporal range and bounding box. The request uses **POST** and configures retries for transient `502/503/504` server responses. The result count is capped to avoid unnecessary pagination when the AOI is broad.

The returned BIOMASS STAC Items are converted into `gdf_biomass` with their footprint geometries and identifiers preserved. As with the NISAR query, this step retrieves metadata footprints only, not raster data.


In [4]:
# ESA MAAP can occasionally return transient 50x errors for large/paginated searches.
# Configure pystac-client to retry those responses, including POST requests.
retry = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=None,
)
stac_io = StacApiIO(max_retries=retry, timeout=60)
biomass_catalog = pystac_client.Client.open(BIOMASS_STAC_URL, stac_io=stac_io)

biomass_search = biomass_catalog.search(
    collections=[BIOMASS_COLLECTION],
    bbox=BBOX,
    datetime=BIOMASS_DT,
    max_items=BIOMASS_MAX_ITEMS,
    method="POST",
)

biomass_items = list(biomass_search.items())
print("BIOMASS items returned:", len(biomass_items))

if not biomass_items:
    raise ValueError(
        "No BIOMASS items were returned. Check BBOX, BIOMASS_DT, and the collection name."
    )

biomass_features = []
for it in biomass_items:
    if it.geometry is None:
        continue
    props = it.properties or {}
    biomass_features.append({
        "type": "Feature",
        "geometry": it.geometry,
        "properties": {
            "biomass_id": it.id,
            "start_datetime": props.get("start_datetime"),
            "end_datetime": props.get("end_datetime"),
            "datetime": props.get("datetime"),
        },
    })

if not biomass_features:
    raise ValueError("BIOMASS items were returned, but none contained usable footprint geometry.")

gdf_biomass = gpd.GeoDataFrame.from_features(biomass_features, crs="EPSG:4326")
gdf_biomass = gdf_biomass[["biomass_id", "start_datetime", "end_datetime", "datetime", "geometry"]]
gdf_biomass.head()


BIOMASS items returned: 200


,biomass_id,start_datetime,end_datetime,datetime,geometry
0,BIO_S1_DGM__1S_20251121T011607_20251121T011628...,2025-11-21T01:16:07.333Z,2025-11-21T01:16:28.063Z,2025-11-21T01:16:07.333Z,"POLYGON ((-139.33798 -78.79231, -137.57281 -78..."
1,BIO_S1_DGM__1S_20251121T011607_20251121T011627...,2025-11-21T01:16:07.529Z,2025-11-21T01:16:27.867Z,2025-11-21T01:16:07.529Z,"POLYGON ((-139.241 -78.78999, -137.48917 -78.9..."
2,BIO_S1_DGM__1S_20251121T011432_20251121T011452...,2025-11-21T01:14:32.087Z,2025-11-21T01:14:52.814Z,2025-11-21T01:14:32.087Z,"POLYGON ((-128.34448 -73.60293, -127.01345 -73..."
3,BIO_S1_DGM__1S_20251121T011432_20251121T011452...,2025-11-21T01:14:32.283Z,2025-11-21T01:14:52.617Z,2025-11-21T01:14:32.283Z,"POLYGON ((-128.3314 -73.59152, -126.99588 -73...."
4,BIO_S1_DGM__1S_20251121T011334_20251121T011355...,2025-11-21T01:13:34.942Z,2025-11-21T01:13:55.669Z,2025-11-21T01:13:34.942Z,"POLYGON ((-124.40576 -70.34375, -123.27435 -70..."


## Interactive map: NISAR and BIOMASS footprint layers

After both metadata queries succeed, this cell creates an interactive Folium map and overlays the two GeoDataFrames using distinct styles (NISAR in blue and BIOMASS in orange). Tooltips show the granule/item identifiers, and the layer control lets you toggle the footprint layers before computing intersections.


In [5]:
# Ensure both footprint queries completed successfully before mapping
if gdf_nisar.empty:
    raise ValueError("gdf_nisar is empty. Re-run the NISAR query with a valid AOI/time range.")
if gdf_biomass.empty:
    raise ValueError("gdf_biomass is empty. Re-run the BIOMASS query with a valid AOI/time range.")

# Ensure both are WGS84 for Folium
gdf_nisar = gdf_nisar.to_crs("EPSG:4326")
gdf_biomass = gdf_biomass.to_crs("EPSG:4326")

# Center/zoom using combined bounds
combined = gpd.GeoSeries(
    list(gdf_nisar.geometry) + list(gdf_biomass.geometry),
    crs="EPSG:4326",
)
minx, miny, maxx, maxy = combined.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=center, tiles="OpenStreetMap", zoom_start=3)

def style_nisar(_):
    return {"color": "#1f77b4", "weight": 2, "fillOpacity": 0.15}

def style_biomass(_):
    return {"color": "#ff7f0e", "weight": 1, "fillOpacity": 0.10}

GeoJson(
    gdf_nisar.__geo_interface__,
    name=f"NISAR ({len(gdf_nisar)})",
    tooltip=folium.GeoJsonTooltip(fields=["nisar_id"]),
    style_function=style_nisar,
).add_to(m)

GeoJson(
    gdf_biomass.__geo_interface__,
    name=f"BIOMASS ({len(gdf_biomass)})",
    tooltip=folium.GeoJsonTooltip(fields=["biomass_id"]),
    style_function=style_biomass,
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m


## Overlap of BIOMASS tiles intersecting with NISAR granules

This cell uses GeoPandas spatial join to identify which BIOMASS footprint polygons intersect which NISAR footprint polygons. It runs gpd.sjoin() with `how="inner"` and `predicate="intersects"` on `gdf_nisar` and `gdf_biomass` (after resetting indices for a clean join), producing pairs where each row represents one intersecting NISAR–BIOMASS match. It then prints the total number of intersection pairs found, and builds a compact summary table called matches by keeping only the `nisar_id` and `biomass_id` columns, removing any duplicate pairs, resetting the index, and showing the first 25 results so you can quickly see which specific granules/tiles overlap.

In [6]:
# Spatial join to find overlapping pairs
pairs = gpd.sjoin(
    gdf_nisar.reset_index(drop=True),
    gdf_biomass.reset_index(drop=True),
    how="inner",
    predicate="intersects",
)

print("Overlap pairs:", len(pairs))

# Compact table of matches (deduped)
matches = pairs[["nisar_id", "biomass_id"]].drop_duplicates().reset_index(drop=True)
matches.head(25)


Overlap pairs: 8


,nisar_id,biomass_id
0,NISAR_L2_PR_GCOV_023_041_D_125_7700_SHNA_A_202...,BIO_S1_DGM__1S_20251121T011451_20251121T011511...
1,NISAR_L2_PR_GCOV_023_041_D_126_7700_SHNA_A_202...,BIO_S1_DGM__1S_20251121T011451_20251121T011511...
2,NISAR_L2_PR_GCOV_023_042_D_127_7700_SHNA_A_202...,BIO_S1_DGM__1S_20251121T025418_20251121T025438...
3,NISAR_L2_PR_GCOV_023_042_D_127_7700_SHNA_A_202...,BIO_S1_DGM__1S_20251121T025358_20251121T025420...
4,NISAR_L2_PR_GCOV_023_042_D_127_7700_SHNA_A_202...,BIO_S1_DGM__1S_20251121T025358_20251121T025419...
5,NISAR_L2_PR_GCOV_023_042_D_128_7700_SHNA_A_202...,BIO_S1_DGM__1S_20251121T025437_20251121T025457...
6,NISAR_L2_PR_GCOV_023_042_D_128_7700_SHNA_A_202...,BIO_S1_DGM__1S_20251121T025437_20251121T025457...
7,NISAR_L2_PR_GCOV_023_042_D_128_7700_SHNA_A_202...,BIO_S1_DGM__1S_20251121T025418_20251121T025438...


This cell creates the actual overlap polygons and then visualizes only those overlaps on a clean map. It first pulls the matching BIOMASS geometries for each join result using pairs`["index_right"]`, wraps them as a GeoSeries aligned to pairs.index, and computes the geometric intersection with the NISAR geometry in each row `(pairs.geometry.intersection(right_geom))`, producing overlap_geom. It then builds a new GeoDataFrame called overlap that keeps just the linked identifiers (nisar_id, biomass_id) plus the intersection geometry, and filters out any empty intersections. For visualization, it creates a fresh Folium map (m_overlap) and adds a single GeoJson layer styled in green with a tooltip showing the two IDs on hover; finally, it automatically zooms the map to the extent of the overlap polygons using `overlap.total_bounds` and `fit_bounds`, so the view centers directly on where overlap occurs without showing the original NISAR/BIOMASS layers.

In [7]:
right_geom = gdf_biomass.loc[pairs["index_right"], "geometry"].values
right_geom = gpd.GeoSeries(right_geom, index=pairs.index, crs="EPSG:4326")

overlap_geom = pairs.geometry.intersection(right_geom)

overlap = gpd.GeoDataFrame(
    pairs[["nisar_id", "biomass_id"]].copy(),
    geometry=overlap_geom,
    crs="EPSG:4326",
)
overlap = overlap[~overlap.geometry.is_empty]

# Map: overlap only
m_overlap = folium.Map(tiles="OpenStreetMap")

def style_overlap(_):
    return {"color": "#2ca02c", "weight": 2, "fillColor": "#2ca02c", "fillOpacity": 0.35}

GeoJson(
    overlap.__geo_interface__,
    name=f"Overlap ({len(overlap)})",
    tooltip=folium.GeoJsonTooltip(fields=["nisar_id", "biomass_id"]),
    style_function=style_overlap,
).add_to(m_overlap)

if len(overlap) > 0:
    minx, miny, maxx, maxy = overlap.total_bounds
    pad = 0.05  
    m_overlap.fit_bounds([[miny - pad, minx - pad], [maxy + pad, maxx + pad]])

m_overlap